In [ ]:
#!/usr/bin/env python3
"""
fetch_from_s3.py
- Reads S3 connection details from env vars (AWS_*).
- Downloads a CSV from s3://$AWS_S3_BUCKET/$TAXI_INPUT_KEY or first CSV under $TAXI_RAW_PREFIX.
- Writes to raw/raw.csv and raw/manifest.json
"""

# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("boto3", "boto3")
_ensure("botocore", "botocore")

# Import libraries
import os
import json
from pathlib import Path
import boto3
from botocore.client import Config

def _get_env(name: str) -> str:
    val = os.getenv(name, "").strip()
    if not val:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return val

def _normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip()
    if not endpoint.startswith("http://") and not endpoint.startswith("https://"):
        endpoint = "http://" + endpoint
    return endpoint

def _s3_client():
    key_id = _get_env("AWS_ACCESS_KEY_ID")
    secret_key = _get_env("AWS_SECRET_ACCESS_KEY")
    region = _get_env("AWS_DEFAULT_REGION")
    endpoint = _normalize_endpoint(_get_env("AWS_S3_ENDPOINT"))

    # Force path-style addressing for broad S3-compatible endpoint compatibility.
    cfg = Config(signature_version="s3v4", s3={"addressing_style": "path"})
    return boto3.client(
        "s3",
        aws_access_key_id=key_id,
        aws_secret_access_key=secret_key,
        region_name=region,
        endpoint_url=endpoint,
        config=cfg,
    )

def _pick_first_csv_key(s3, bucket: str, prefix: str) -> str:
    prefix = prefix if prefix.endswith("/") else prefix + "/"
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            k = obj["Key"]
            if k.lower().endswith(".csv"):
                keys.append(k)
    if not keys:
        raise RuntimeError(f"No CSV found under s3://{bucket}/{prefix}")
    return sorted(keys)[0]

def main() -> None:
    s3 = _s3_client()
    bucket = _get_env("AWS_S3_BUCKET")

    raw_prefix = os.getenv("TAXI_RAW_PREFIX", "raw_data/").strip() or "raw_data/"
    input_key = os.getenv("TAXI_INPUT_KEY", "").strip()

    key = input_key if input_key else _pick_first_csv_key(s3, bucket, raw_prefix)

    out_csv = Path(os.getenv("TAXI_FETCH_OUT", "raw/raw.csv"))
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    print(f"Downloading s3://{bucket}/{key} -> {out_csv}", flush=True)
    s3.download_file(bucket, key, str(out_csv))

    manifest = {
        "source_bucket": bucket,
        "source_key": key,
        "source_uri": f"s3://{bucket}/{key}",
        "raw_prefix": raw_prefix,
    }
    manifest_path = Path("raw/manifest.json")
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print(f"Wrote {out_csv} ({out_csv.stat().st_size} bytes)", flush=True)
    print(f"Wrote {manifest_path}", flush=True)

if __name__ == "__main__":
    main()